[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-skill_suggestion/examples/skill_suggestion/skill_suggestion.ipynb)

# Suggest one intent from a large catalog

A catalog of intents does not fit a polite prompt, and the right answer is often "none of these". FRIDA-Decisions first ranks the catalog in one request — option ids are not shown to the model — then a second request asks a yes/no question of each of the top candidates. Code keeps the candidate with the highest yes-probability, and abstains when that probability is below a cutoff.

`FAST = True` ranks a pool of 40 intents that includes every labelled one. Set it to `False` (or pass `--full` to the script) to rank all 243 intents in `examples/data/intent_catalog.json`. The state of a long dialogue can hit the default 384-token cut; `usage.state_truncated` reports that.

Pattern: [Skill suggestion](https://docs.typesafe.ai/cookbooks/skill_suggestion).


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
TOP_K = 5
NOUL_MIN = 0.5
FAST_POOL = 40


def pool_for(criteria: dict, gold: list[str], fast: bool) -> dict:
    if not fast:
        return criteria
    chosen = list(dict.fromkeys([*gold, *criteria]))[:FAST_POOL]
    return {key: criteria[key] for key in chosen}


def suggest(judge, text: str, criteria: dict) -> dict:
    ranked = judge({
        "state": text,
        "questions": {"best": {
            "type": "ranking",
            "instructions": "Какое намерение лучше всего описывает сообщение клиента?",
            "criteria": criteria,
        }},
    })
    top = ranked["answers"]["best"]["ranking"][:TOP_K]
    checked = judge({
        "state": text,
        "questions": {
            key: {"type": "noul", "instructions": "Клиент хочет именно это: " + criteria[key]}
            for key in top
        },
    })
    # A near-tie keeps the ranking's order. int8 ONNX can swap two 0.98s.
    best_noul = max(checked["answers"][key]["noul"] for key in top)
    close = [key for key in top if best_noul - checked["answers"][key]["noul"] <= 0.02]
    best = close[0]
    noul = checked["answers"][best]["noul"]
    return {
        "intent": best if noul >= NOUL_MIN else None,
        "noul": noul,
        "top": [(key, checked["answers"][key]["noul"]) for key in top],
        "truncated": ranked["usage"]["state_truncated"] or checked["usage"]["state_truncated"],
        "ranked_ms": ranked["usage"]["milliseconds"],
    }


## Rank, then re-check

The ranking always names a winner, because its probabilities split one unit across the list.
The yes/no pass can refuse all of them. With `fast=True` the pool is 40 intents, not 243.


In [4]:
def run(judge, rows: list[dict], catalog_path, fast: bool = True) -> None:
    catalog = json.loads(Path(catalog_path).read_text(encoding="utf-8"))
    criteria = catalog["request"]["questions"]["intent"]["criteria"]
    picked = pool_for(criteria, [row["intent"] for row in rows], fast)
    print(f"pool {len(picked)} of {len(criteria)} intents, top-{TOP_K}, noul cutoff {NOUL_MIN}")
    hits = 0
    for row in rows:
        result = suggest(judge, row["text"], picked)
        hits += result["intent"] == row["intent"]
        top = " ".join(f"{key}:{noul:.2f}" for key, noul in result["top"])
        print(f"{'ok' if result['intent'] == row['intent'] else 'miss':4} "
              f"got {result['intent']!s:28} gold {row['intent']:28} "
              f"noul {result['noul']:.2f}  {result['ranked_ms']:.0f} ms")
        print(f"     top {top}")
        if result["truncated"]:
            print("     state truncated: default cut is 384 tokens")
    print(f"\n{hits}/{len(rows)} match. fast={fast}")


In [5]:
FAST = True

def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-skill-suggestion/examples/skill_suggestion/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

def catalog_file() -> Path:
    import urllib.request
    for path in (Path("../data/intent_catalog.json"), Path("examples/data/intent_catalog.json"),
                 Path("intent_catalog.json")):
        if path.exists():
            return path
    dest = Path("intent_catalog.json")
    url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
           "main/examples/data/intent_catalog.json")
    dest.write_bytes(urllib.request.urlopen(url, timeout=120).read())
    return dest

run(judge, load_rows(), catalog_file(), fast=FAST)


pool 40 of 243 intents, top-5, noul cutoff 0.5
ok   got port_in_keep_number          gold port_in_keep_number          noul 0.98  3798 ms
     top port_in_keep_number:0.98 tariff_change:0.94 new_sim_new_number:0.98 change_number_keep_tariff:0.96 port_out_to_other_operator:0.86
ok   got sim_lost_block               gold sim_lost_block               noul 1.00  3576 ms
     top sim_lost_block:1.00 number_restore_after_block:0.27 sim_replace:0.24 close_number:0.64 esim_transfer_new_phone:0.09
ok   got tariff_current_info          gold tariff_current_info          noul 0.99  3574 ms
     top tariff_current_info:0.99 tariff_change_cost:0.79 tariff_choose:0.88 tariff_minutes_left:0.87 tariff_change:0.31
ok   got esim_activate                gold esim_activate                noul 0.98  3883 ms
     top esim_activate:0.98 esim_transfer_new_phone:0.95 new_sim_new_number:0.96 sim_for_smartwatch:0.73 sim_replace:0.96

4/4 match. fast=True
